In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv
from rich import print
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage
from langchain.chat_models import init_chat_model

In [2]:
load_dotenv()
MODEL = os.getenv("GROQ_MODEL","groq:qwen/qwan3.8-27b")

In [3]:
RESUME_DIR = ( Path().cwd().parent / "resumes" )

In [4]:
MODEL

'groq:qwen/qwen3.8-27b'

In [5]:
@tool
def add(a:float,b:float) -> float:
    """Add two number together and return the addition result"""
    return a + b

In [6]:
def get_weather(city:str) -> str:
    """Get the current weather of given city name"""
    weather = {"mumbai":"32C , humid" , "delhi":"38C , hazy" , "bengaluru":"26C , plesanat"}
    return f"City {city} - {weather.get(city.lower(),f"No weather data for {city}")}"

In [7]:
get_weather("mumbai")

'City mumbai - 32C , humid'

In [8]:
get_weather("bela")

'City bela - No weather data for bela'

In [9]:
get_weather("Mumbai")

'City Mumbai - 32C , humid'

In [10]:
@tool
def get_weather(city:str) -> str:
    """Get the current weather of given city name"""
    weather = {"mumbai":"32C , humid" , "delhi":"38C , hazy" , "bengaluru":"26C , plesanat"}
    return f"City {city} - {weather.get(city.lower(),f"No weather data for {city}")}" 

In [11]:
get_weather

StructuredTool(name='get_weather', description='Get the current weather of given city name', args_schema=<class 'langchain_core.utils.pydantic.get_weather'>, func=<function get_weather at 0x000001E187B014E0>)

In [12]:
get_weather.name

'get_weather'

In [13]:
get_weather.description

'Get the current weather of given city name'

In [14]:
@tool
def lookup_user(user_id:str) -> str:
    """Look up a user's name and plan by their id(eg.'u1')."""
    users = {"u1":"Aarav (Pro plan)","u2":"Divya (Free plan)"}
    return users.get(user_id, f"No user with id '{user_id}'.")

In [15]:
def get_resume(name:str) -> str:
    """Read and the return stored resume for candidate name (ramesh, khushi)"""
    first_name = name.lower().strip().split()[0]
    file_path = RESUME_DIR / f"{first_name}.txt"
    if file_path.exists():
        return file_path.read_text(encoding="utf-8")
    return f"No resume found for {name}"

In [16]:
get_resume("priya")

'Name: Priya Singh\nRole: DevOps Engineer\nExperience: 5 years\nSkills: AWS, Kubernetes, Docker, Terraform, CI/CD (GitHub Actions), Linux\nBackground:\n5 years of hands-on experience in cloud infrastructure and automating deployment pipelines. Expert in AWS services, managing Kubernetes clusters, and infrastructure-as-code with Terraform.'

In [17]:
get_resume("raju")

'No resume found for raju'

In [18]:
print(get_resume("priya"))

Name: Priya Singh
Role: DevOps Engineer
Experience: 5 years
Skills: AWS, Kubernetes, Docker, Terraform, CI/CD (GitHub Actions), Linux
Background:
5 years of hands-on experience in cloud infrastructure and automating deployment pipelines. Expert in AWS services,
managing Kubernetes clusters, and infrastructure-as-code with Terraform.

In [19]:
@tool
def get_resume(name:str) -> str:
    """Read and the return stored resume for candidate name (ramesh, khushi)"""
    first_name = name.lower().strip().split()[0]
    file_path = RESUME_DIR / f"{first_name}.txt"
    if file_path.exists():
        return file_path.read_text(encoding="utf-8")
    return f"No resume found for {name}"

In [20]:
get_resume

StructuredTool(name='get_resume', description='Read and the return stored resume for candidate name (ramesh, khushi)', args_schema=<class 'langchain_core.utils.pydantic.get_resume'>, func=<function get_resume at 0x000001E187C1A3E0>)

In [21]:
TOOLS = [add,get_weather,lookup_user,get_resume]

In [22]:
TOOLS_DATA = {
                "add":add,
                "get_weather":get_weather,
                "lookup_user":lookup_user,
                "get_resume":get_resume
            }

In [23]:
TOOLS_DATA["add"]

StructuredTool(name='add', description='Add two number together and return the addition result', args_schema=<class 'langchain_core.utils.pydantic.add'>, func=<function add at 0x000001E18799ECA0>)

In [24]:
llm = init_chat_model(MODEL, temperature=0)
llm_with_tools = llm.bind_tools(TOOLS)

In [25]:
prompt = "can we select priya for python developer interview"
messages=[HumanMessage(content=prompt)]
print(f"[bold cyan] USER: [/bold cyan] {prompt}\n")

 USER:  can we select priya for python developer interview

In [26]:
messages

[HumanMessage(content='can we select priya for python developer interview', additional_kwargs={}, response_metadata={})]

In [27]:
ai = llm_with_tools.invoke(messages)

In [28]:
ai

AIMessage(content='I don\'t have any information about a candidate named "Priya" in my available tools. Let me check if there\'s a resume stored for her.\n\n', additional_kwargs={'tool_calls': [{'id': 't24m25r37', 'function': {'arguments': '{"name":"priya"}', 'name': 'get_resume'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 58, 'prompt_tokens': 488, 'total_tokens': 546, 'completion_time': 0.132001538, 'completion_tokens_details': None, 'prompt_time': 0.03572106, 'prompt_tokens_details': None, 'queue_time': 0.056018679, 'total_time': 0.167722598}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_424cb89518', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0f799-efae-77c3-aa3f-ec17f108201f-0', tool_calls=[{'name': 'get_resume', 'args': {'name': 'priya'}, 'id': 't24m25r37', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 488, 'output_tokens':

In [ ]:
result.additional_kwargs["tool_calls"]

[{'id': '48kg3t70w',
  'function': {'arguments': '{"name":"priya"}', 'name': 'get_resume'},
  'type': 'function'}]

In [30]:
messages.append(ai)

In [31]:
messages

[HumanMessage(content='can we select priya for python developer interview', additional_kwargs={}, response_metadata={}),
 AIMessage(content='I don\'t have any information about a candidate named "Priya" in my available tools. Let me check if there\'s a resume stored for her.\n\n', additional_kwargs={'tool_calls': [{'id': 't24m25r37', 'function': {'arguments': '{"name":"priya"}', 'name': 'get_resume'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 58, 'prompt_tokens': 488, 'total_tokens': 546, 'completion_time': 0.132001538, 'completion_tokens_details': None, 'prompt_time': 0.03572106, 'prompt_tokens_details': None, 'queue_time': 0.056018679, 'total_time': 0.167722598}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_424cb89518', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0f799-efae-77c3-aa3f-ec17f108201f-0', tool_calls=[{'name': 'get_resume', 'args': {'name': 'priya

In [32]:
ai.tool_calls

[{'name': 'get_resume',
  'args': {'name': 'priya'},
  'id': 't24m25r37',
  'type': 'tool_call'}]

In [33]:
ai.content

'I don\'t have any information about a candidate named "Priya" in my available tools. Let me check if there\'s a resume stored for her.\n\n'

In [34]:
ai

AIMessage(content='I don\'t have any information about a candidate named "Priya" in my available tools. Let me check if there\'s a resume stored for her.\n\n', additional_kwargs={'tool_calls': [{'id': 't24m25r37', 'function': {'arguments': '{"name":"priya"}', 'name': 'get_resume'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 58, 'prompt_tokens': 488, 'total_tokens': 546, 'completion_time': 0.132001538, 'completion_tokens_details': None, 'prompt_time': 0.03572106, 'prompt_tokens_details': None, 'queue_time': 0.056018679, 'total_time': 0.167722598}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_424cb89518', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0f799-efae-77c3-aa3f-ec17f108201f-0', tool_calls=[{'name': 'get_resume', 'args': {'name': 'priya'}, 'id': 't24m25r37', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 488, 'output_tokens':

In [40]:
tool_name = ai.tool_calls[0]["name"]
tool_ag = ai.tool_calls[0]["args"]["name"]
tool_id = ai.tool_calls[0]["id"]
resume = get_resume.invoke(tool_ag)

print(resume)

Name: Priya Singh
Role: DevOps Engineer
Experience: 5 years
Skills: AWS, Kubernetes, Docker, Terraform, CI/CD (GitHub Actions), Linux
Background:
5 years of hands-on experience in cloud infrastructure and automating deployment pipelines. Expert in AWS services,
managing Kubernetes clusters, and infrastructure-as-code with Terraform.

In [38]:
tool_name , tool_ag

('get_resume', 'priya')

In [41]:
messages.append(ToolMessage(content=resume, tool_call_id=tool_id))

In [42]:
messages

[HumanMessage(content='can we select priya for python developer interview', additional_kwargs={}, response_metadata={}),
 AIMessage(content='I don\'t have any information about a candidate named "Priya" in my available tools. Let me check if there\'s a resume stored for her.\n\n', additional_kwargs={'tool_calls': [{'id': 't24m25r37', 'function': {'arguments': '{"name":"priya"}', 'name': 'get_resume'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 58, 'prompt_tokens': 488, 'total_tokens': 546, 'completion_time': 0.132001538, 'completion_tokens_details': None, 'prompt_time': 0.03572106, 'prompt_tokens_details': None, 'queue_time': 0.056018679, 'total_time': 0.167722598}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_424cb89518', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0f799-efae-77c3-aa3f-ec17f108201f-0', tool_calls=[{'name': 'get_resume', 'args': {'name': 'priya

In [43]:
result_final = llm_with_tools.invoke(messages)

In [44]:
result_final

AIMessage(content="Based on Priya Singh's resume, I would **not recommend** selecting her for a **Python Developer** interview. Here's why:\n\n- **Role:** She is a **DevOps Engineer**, not a Python Developer.\n- **Skills:** Her expertise is in **AWS, Kubernetes, Docker, Terraform, CI/CD, and Linux** — all infrastructure and operations-focused technologies.\n- **Experience:** Her 5 years of experience are in **cloud infrastructure and deployment automation**, not in Python application development.\n\nWhile DevOps engineers often use Python for scripting and automation, her profile is clearly oriented toward **infrastructure and operations**, not software development in Python.\n\nIf you're looking for a Python Developer, you'd want a candidate with strong experience in Python frameworks (e.g., Django, Flask, FastAPI), data structures, algorithms, and application development.\n\nWould you like me to look up another candidate?", additional_kwargs={}, response_metadata={'token_usage': {'co

In [45]:
print(result_final.content)

Based on Priya Singh's resume, I would **not recommend** selecting her for a **Python Developer** interview. Here's
why:

- **Role:** She is a **DevOps Engineer**, not a Python Developer.
- **Skills:** Her expertise is in **AWS, Kubernetes, Docker, Terraform, CI/CD, and Linux** — all infrastructure and
operations-focused technologies.
- **Experience:** Her 5 years of experience are in **cloud infrastructure and deployment automation**, not in 
Python application development.

While DevOps engineers often use Python for scripting and automation, her profile is clearly oriented toward 
**infrastructure and operations**, not software development in Python.

If you're looking for a Python Developer, you'd want a candidate with strong experience in Python frameworks (e.g.,
Django, Flask, FastAPI), data structures, algorithms, and application development.

Would you like me to look up another candidate?